# Azure Databricks에서 워크로드 모니터링, 문제 해결 및 최적화

## 클러스터 소비 모니터링 및 관리

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.monitor-manage-cluster-consumption.png)

### 클러스터 소비의 영향 이해
> **트레이너 데모 가이드**
>> 1. **계정 콘솔** > **사용**으로 이동하여 전반적인 DBU 사용 추세를 확인하세요.
>> 2. 근무 시간 사이 차트의 평평한 선을 통해 유휴 클러스터의 예를 지적합니다.
>> 3. 주말 또는 야간 사용량 급증을 모니터링되지 않는 비용의 예로 보여줍니다.

>> **주요 메시지:** 목표는 단순히 비용을 최소화하는 것이 아니라 **워크로드 요구 사항에 정확하게 리소스를 일치시키는 것**입니다.
---

### 컴퓨팅 지표 모니터링 및 정렬
> **트레이너 데모 가이드**
>> 1. 사이드바에서 **컴퓨팅**으로 이동하여 클러스터를 선택하고 **메트릭** 탭을 클릭합니다.
>> 2. 세 가지 측정항목 카테고리를 살펴보세요.
>>    - **하드웨어 측정항목**: CPU 활용도, 메모리 활용도, 네트워크 처리량
>>    - **Spark 지표**: 활성 작업, 실패/완료된 작업, 읽기/쓰기 셔플
>>    - **GPU 측정항목**(Databricks Runtime ML 13.3 이상에서 사용 가능)
>> 3. 날짜 선택기를 사용하여 지난 30일 동안의 데이터를 필터링합니다.
>> 4. **컴퓨팅** 드롭다운에서 개별 노드를 선택하여 작업자 성능을 비교합니다.
>> 5. 각 측정항목 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 값을 오름차순 또는 내림차순으로 정렬하세요.
>> **주요 신호:** 높은 셔플 읽기/쓰기 값은 조사할 가치가 있는 데이터 이동 병목 현상을 나타냅니다.
>> 아래 코드 셀을 실행하여 시스템 테이블에서 최근 클러스터 메타데이터를 검사합니다.

In [0]:
%sql
-- === DEMO: Inspect recent cluster metadata from system tables ===
-- system.compute.clusters contains metadata about all clusters in your account

SELECT
  *
FROM system.compute.clusters
LIMIT 20;

### SQL 웨어하우스 성능 모니터링
> **트레이너 데모 가이드**
>> 1. 왼쪽 사이드바의 **SQL Warehouse**로 이동합니다.
>> 2. 창고를 선택하고 **모니터링** 탭을 엽니다.
>> 3. 실시간 통계 패널을 살펴보세요.
>>    - **상태**, **실행 중인 쿼리**, **대기 중인 쿼리**, **클러스터 수**
>> 4. **최대 쿼리 수** 차트를 표시합니다. 급증은 수요가 높은 기간을 나타냅니다.
>> 5. **실행 중인 클러스터** 차트 표시 - 자동 확장 동작 실행 중
>> 6. **쿼리 기록**에서 장기 실행 쿼리를 필터링하고, 열 이름 위에 마우스를 올려 정렬 아이콘을 클릭하여 값을 오름차순 또는 내림차순으로 정렬합니다. 리소스 영향에 대해 논의합니다.

>> **핵심 사항:**
>> - 클러스터 수가 장기간 최대값으로 유지되는 경우 최대 클러스터 한도를 늘립니다.
>> - 쿼리 기록을 사용하여 불균형한 리소스를 소비하는 사용자 또는 쿼리를 식별합니다.

### 자동 종료 및 자동 크기 조정 구성
> **트레이너 데모 가이드**
>> **자동 종료:**
> 1. **컴퓨팅**으로 이동 > 다목적 클러스터 편집
> 2. 개발 클러스터의 경우 **자동 종료**를 활성화하고 **30~60분**으로 설정합니다.
> 3. 클러스터는 *모든* 활동이 중지될 때만 종료됩니다(Spark 작업, 구조적 스트리밍, JDBC 호출).
>> **자동 확장:**
> 1. **Autoscaling** 토글을 활성화하고 최소(예: 2) 및 최대(예: 8) 작업자 노드를 설정합니다.
> 2. Databricks 자동 조정 - 고정 크기 클러스터에 비해 일반적으로 **20~40%** 절감
> 3. 서버리스 컴퓨팅과 대조: 서버리스는 자동으로 0으로 확장됩니다.
>> **중요:** 자동 종료는 DStream을 모니터링하지 않습니다. DStream 워크로드를 사용하는 경우 자동 종료를 비활성화하거나 구조적 스트리밍으로 마이그레이션하세요.

---
#### 정렬
> **정렬 방법:** 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하거나, SQL 쿼리에서 `ORDER BY` 또는 PySpark에서 `.sort()`/`.orderBy()`를 사용하여 원하는 기준으로 정렬할 수 있습니다.

### 예산 및 시스템 테이블을 사용하여 비용 추적
> **트레이너 데모 가이드**
>> **예산(계정 콘솔):**
> 1. **계정 콘솔** > **사용** > **예산**으로 이동합니다.
> 2. **예산 추가**를 클릭하고 지출 한도를 구성하고 이메일 알림을 설정합니다.
> 3. **계정 관리자** 권한이 필요합니다.
>> **비용 분석을 위한 시스템 테이블:**
> - `system.billing.usage` -- 클러스터, 작업 및 SKU당 DBU 소비
> - `system.compute.clusters` -- 소유자를 포함한 클러스터 메타데이터
> - 누가 무엇을 지출하는지 이해하려면 다음 테이블을 결합하세요.
>> **정렬:** 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하거나, SQL 쿼리에서 `ORDER BY` 또는 `SORT BY` 절을 사용하여 원하는 기준으로 정렬할 수 있습니다.
>> **팁:** 처음부터 클러스터와 작업 영역에 태그(예: `business_unit`, `project`, `environment`)를 적용하세요. 태그는 청구 기록에 전파되어 정확한 지불 거절을 가능하게 합니다.
>> 아래 SQL 쿼리를 실행하여 실제 비용 분석을 보여줍니다.

In [0]:
%sql

-- === DEMO: Daily DBU consumption trend ===
-- See how compute usage changes day by day -- spot unusual spikes or weekly patterns

SELECT
  usage_date,
  SUM(usage_quantity) AS DBUs_Consumed
FROM system.billing.usage
WHERE sku_name = 'STANDARD_ALL_PURPOSE_COMPUTE'
GROUP BY usage_date
ORDER BY usage_date ASC;

In [0]:
%sql

-- === DEMO: Which jobs consumed the most DBUs? ===
-- Identify the biggest cost drivers among scheduled jobs

SELECT
  usage_metadata.job_id AS job_id,
  SUM(usage_quantity) AS total_dbus
FROM system.billing.usage
WHERE usage_metadata.job_id IS NOT NULL
GROUP BY job_id
ORDER BY total_dbus DESC
LIMIT 20;

In [0]:
%sql

-- === DEMO: Attribute costs to cluster owners ===
-- Identify which team members or service accounts drive the most resource consumption

SELECT
  c.owned_by,
  SUM(u.usage_quantity) AS total_dbus
FROM system.billing.usage u
JOIN system.compute.clusters c
  ON u.usage_metadata.cluster_id = c.cluster_id
WHERE u.usage_metadata.cluster_id IS NOT NULL
GROUP BY c.owned_by
ORDER BY total_dbus DESC;

In [0]:
%sql

-- === DEMO: Usage by custom tag ===
-- Break down costs by business unit, project, or environment tag

SELECT
  sku_name,
  usage_unit,
  custom_tags['project'] AS project_tag,
  SUM(usage_quantity) AS total_usage
FROM system.billing.usage
WHERE custom_tags['project'] IS NOT NULL
GROUP BY sku_name, usage_unit, custom_tags['project']
ORDER BY total_usage DESC;

## Lakeflow 작업 문제 해결 및 복구

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.troubleshoot-repair-lakeflow-jobs.png)

### 작업 실패 상태 이해
Lakeflow 작업 실행이 종료되면 다음 상태 중 하나가 될 수 있습니다.

| State                  | Meaning                                                      |
|------------------------|-------------------------------------------------------------|
| **Failed**             | One or more leaf tasks didn't complete successfully         |
| **Succeeded with failures** | Some tasks failed, but all leaf tasks completed         |
| **Skipped**            | Job exceeded maximum concurrent runs                        |
| **Timed Out**          | Job exceeded configured maximum duration                    |
| **Canceled**           | Stopped by user or automated process                        |

---

### 실패 원인 파악
> **트레이너 데모 가이드**
>> 1. 사이드바에서 **작업 및 파이프라인**으로 이동합니다.
> 2. 작업을 선택하고 **실행** 탭을 엽니다.
> 3. **매트릭스 보기**를 표시합니다. 각 셀은 여러 실행에 대한 작업 상태입니다.
> 4. 빨간색(실패한) 작업 위로 마우스를 가져가면 시작 시간, 종료 시간, 상태, 오류 메시지를 확인하세요.
> 5. 실패한 작업을 클릭하여 전체 출력 및 로그가 포함된 **작업 실행 세부 정보**를 엽니다.
> 6. **오류 진단**을 클릭하여 제안을 위해 Genie Code를 호출합니다.
>> **패턴 인식:**
> - 동일한 작업이 반복적으로 실패합니다. -> 해당 작업과 관련된 코드 또는 구성 문제
> - 다양한 작업 전반에 걸친 무작위 오류 -> 클러스터 또는 리소스 문제

In [0]:
# === DEMO: Simulate a task-level data quality failure ===
# Demonstrates how schema/type issues surface and how to detect them

from pyspark.sql.functions import col

# Dataset with a mixed-type 'amount' column
data = [
    (1, 'Alice', '150.00'),     # valid
    (2, 'Bob',   'not_a_num'),  # invalid -- cast produces NULL by default
    (3, 'Carol', '275.50'),     # valid
    (4, 'Dave',  None),         # null
]
df = spark.createDataFrame(data, ['id', 'name', 'amount_str'])

# Default: invalid casts produce NULL (no exception)
df_cast = df.withColumn('amount', col('amount_str').cast('double'))
df_cast.show()

# Detect rows that failed the cast (practical quarantine pattern)
bad_rows = df_cast.filter(col('amount').isNull() & col('amount_str').isNotNull())
print(f'Rows failed cast: {bad_rows.count()}')
bad_rows.show()

# To raise a hard failure visible in the Spark UI, enable ANSI mode:
# spark.conf.set('spark.sql.ansi.enabled', 'true')
# spark.sql("SELECT CAST('not_a_num' AS DOUBLE)").show()

### 실패한 작업 복구 실행 실행
> **트레이너 데모 가이드**
>> 1. **실행** 탭의 **시작 시간** 열에서 실패한 실행을 클릭합니다.
>> 2. 오른쪽 상단의 **Repair run**을 클릭하세요.
>> 3. 다시 실행할 작업 목록을 검토합니다(실패한 작업 + 해당 종속 항목).
>> 4. 선택적으로 복구 대화 상자에서 작업 매개변수를 수정합니다.
>> 5. **Repair run**을 클릭하여 재실행을 시작합니다.
>> **핵심 사항:**
>> - 실패한 작업과 그 종속 작업만 다시 실행됩니다. - 시간을 절약하고 비용을 계산합니다.
>> - 수리가 완료된 후 매트릭스 보기에 새 열이 추가됩니다.
>> - 수리는 **2개 이상의 작업**이 있는 작업에만 가능합니다.
>> **정렬:** 실행 탭에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 실행을 오름차순 또는 내림차순으로 정렬할 수 있습니다.
>> **중지/다시 시작 제어:**
>> - **중지**: 실행 탭에서 활성 실행에 대한 중지 버튼을 클릭합니다.
>> - **연속 작업 다시 시작**: **실행 다시 시작**을 사용하여 지수 백오프 우회 반복적인 실패 이후

### 작업 상태를 사전에 모니터링
> **트레이너 데모 가이드**
>> **작업 알림 구성:**
> 1. 채용 공고 열기 > **수정** 클릭 > **알림**으로 스크롤
> 2. 시작, 성공, 실패, 기간 초과에 대한 이메일, Slack 또는 웹훅 알림 추가
> 3. **최대 재시도** 및 **재시도 간격**을 구성할 위치를 표시합니다.
>> **정렬:** 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하거나, SQL 쿼리에서 `ORDER BY` 또는 `SORT BY` 절을 사용하여 원하는 기준으로 정렬할 수 있습니다.
>> **모범 사례:**
> - 정기적으로 **매트릭스 보기**를 검토하세요. 특정 작업의 실패 패턴은 주의가 필요한 영역을 나타냅니다.
> - `system.lakeflow` 테이블을 사용하여 운영 팀을 위한 작업 공간 간 대시보드 구축
> - 비용 폭주를 방지하기 위해 장기 실행 작업에 **최대 기간**을 설정하세요.

## Spark 작업 및 노트북 문제 해결

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.troubleshoot-repair-spark-jobs-notebooks.png)

### Spark 작업 실패의 일반적인 원인 이해
Spark 작업 실패는 일반적으로 세 가지 범주로 분류됩니다.
 Category | Examples | First Signal |
----------|---------|--------------|
 **Code errors** | Syntax errors, schema mismatches, data quality issues | Error message with stack trace |
 **Resource bottlenecks** | Out-of-memory (OOM), slow shuffle, spill to disk | Long task duration, executor failures |
 **Environmental issues** | Cluster startup failure, spot instance reclamation | Event log entries |

---

### Spark UI를 사용하여 문제 진단 및 정렬
> **트레이너 데모 가이드**
>> 1. 클러스터 > **Spark UI** 탭 > **작업** 페이지로 이동합니다.
>> 2. **작업 타임라인**에서 세 가지 패턴을 살펴보세요.
> - **실패한 작업**: 빨간색 상태 -- 실패한 단계 및 오류를 보려면 클릭하세요.
> - **장기 실행 작업**: 타임라인을 지배하는 단일 작업 - 해당 단계를 확인하세요.
> - **실행 공백**: Spark 작업이 발생하지 않음 - 드라이버가 과부하될 수 있음
>> 3. 장기 실행 작업 클릭 > **단계** > 가장 긴 단계 식별
>> 4. 단계 세부정보에서 **요약 측정항목**을 검토합니다.
> - **입력/출력**: 데이터 읽기/쓰기
> - **Shuffle Read/Write**: 노드 간에 데이터 이동
> - **작업 기간 분포**: 이상값 작업 찾기(데이터 편향)
>> **정렬:** 단계 세부정보에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 작업을 오름차순 또는 내림차순으로 정렬할 수 있습니다.
>> **주요 통찰력:** 작업이 1개뿐인 단계 = 병렬성이 부족합니다. 고르지 못한 작업 기간 = 데이터 편향.
>> 아래 코드 셀을 실행하여 Spark UI에서 검사할 다단계 Spark 작업을 생성합니다.

In [0]:
# === DEMO: Generate a multi-stage Spark job to analyze in the Spark UI ===
# Creates visible stages, shuffle, and aggregation patterns

from pyspark.sql.functions import col, sum as spark_sum, count

print('Generating data for Spark UI analysis...')

# Stage 1: create base data
df = (spark.range(0, 500_000)
        .toDF('id')
        .withColumn('category', (col('id') % 100).cast('string'))
        .withColumn('value',    (col('id') * 3.14).cast('double')))

# Stage 2: aggregation triggers a shuffle (sort-merge or hash aggregate)
agg_df = df.groupBy('category').agg(
    count('*').alias('record_count'),
    spark_sum('value').alias('total_value')
)

# Stage 3: sort triggers another shuffle
result = agg_df.orderBy('total_value', ascending=False)
result.display()

print('\nJob complete.')
print('Navigate to: Compute > Your Cluster > Spark UI > Jobs')
print('Examine the three stages, shuffle read/write sizes, and task distribution.')

### 리소스 병목 현상 식별 및 해결
> **트레이너 데모 가이드** -- Spark UI: 측정항목 탭>> **메모리 부족:**> - 높은 메모리 활용도 + **Shuffle Spill(디스크) > 0** 단계 지표> - 수정: 작업자 인스턴스 크기 늘리기, 파티션 수 줄이기, 변환 최적화>> **CPU 제약:**> - 긴 작업 실행 시간으로 인한 높은 CPU 사용률> - 수정: 호환되는 워크로드에 대해 **Photon** 활성화, 작업자 노드 추가>> **네트워크 병목 현상:**> - 실행 속도가 느린 대용량 **Shuffle Read/Write** 볼륨> - 수정: 데이터를 더 일찍 필터링하고, 브로드캐스트 조인을 사용하고, 불필요한 재파티션을 줄입니다.>> **서버 부하 분산(메트릭 탭):**> - 빨간색 = 과부하, 파란색 = 유휴 상태> - 빨간색 드라이버 + 파란색 작업자 -> 드라이버가 과부하되어 더 큰 인스턴스 유형이 필요함> - **컴퓨팅** > 클러스터 > **메트릭** > 서버 부하 분산으로 이동합니다.

### 클러스터를 다시 시작하여 환경 문제 해결
> **트레이너 데모 가이드**
>> **다시 시작하기 전에 이벤트 로그를 확인하세요.**
> 1. **컴퓨팅** > 클러스터 선택 > **이벤트 로그** 탭
> 2. 검색: `DRIVER_NOT_RESPONDING`, 스팟 인스턴스 회수, 실행기 종료
>> **UI를 통해 다시 시작:**
> 1. **컴퓨팅** > 클러스터 선택
> 2. **다시 시작**을 클릭하고 확인하면 실행 중인 모든 작업이 종료됩니다.
>> **Databricks CLI를 통해 다시 시작:**
> bash
> # CLUSTER_ID를 찾기 위해 클러스터를 나열합니다.
> databricks clusters list
>
> # 클러스터 다시 시작
> databricks clusters restart --cluster-id <CLUSTER_ID>
> 
>> **중요:** 다시 시작하면 실행 중인 작업이 종료되고 Spark UI 기록이 재설정됩니다. 다시 시작하기 전에 진단 정보를 저장하십시오.

---

#### 결과 테이블 정렬 방법
- 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 오름차순 또는 내림차순으로 정렬할 수 있습니다.
- SQL 쿼리에서는 `ORDER BY` 또는 PySpark에서는 `.orderBy()` 또는 `.sort()` 메서드를 사용하여 원하는 기준으로 정렬할 수 있습니다.

## 캐싱, 왜곡, 유출, 셔플 조사

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.resolve-caching-skewing-spilling-shuffle.png)

### 진단 도구를 사용하여 문제 식별
Azure Databricks는 성능 문제를 조사하기 위한 세 가지 기본 도구를 제공합니다.
| Tool | Best for | Access |
|------|----------|--------|
| **Query DAG** | Understanding execution flow, operator bottlenecks | Spark UI > SQL/DataFrame tab |
| **Spark UI** | Stage/task metrics, shuffle volumes, spill, task skew | Compute > Cluster > Spark UI |
| **Query Profile** | SQL warehouse analysis, per-operator timing | SQL Editor > Query History > See query profile |

> **팁:** DAG로 시작하여 전체 흐름을 이해한 다음 Spark UI 또는 쿼리 프로필을 사용하여 특정 느린 단계를 자세히 살펴보세요.

In [0]:
# === DEMO: Check disk cache (Delta cache) and Spark cache settings ===

# 1. Check if disk cache is enabled (automatically caches Parquet files on worker SSDs)
disk_cache = spark.conf.get('spark.databricks.io.cache.enabled')
print(f'Disk cache (Delta cache) enabled: {disk_cache}')

# 2. Demonstrate Spark in-memory cache with .cache() and .unpersist()
from pyspark.sql.functions import col
import time

df = (spark.range(0, 500_000)
        .withColumn('value',    (col('id') * 2.5).cast('double'))
        .withColumn('category', (col('id') % 20).cast('string')))

# Cache the DataFrame in memory
df.cache()

# First action: materializes and caches
t0 = time.time(); df.count(); t1 = time.time()
print(f'First count  (cache miss): {t1 - t0:.2f}s')

# Second action: served from cache -- should be faster
t0 = time.time(); df.count(); t1 = time.time()
print(f'Second count (cache hit):  {t1 - t0:.2f}s')

# IMPORTANT: free memory when done
df.unpersist()
print('Cache cleared with .unpersist()')
print('\nCheck Spark UI > Storage tab to see currently cached DataFrames.')

In [0]:
# === DEMO: Investigate data skew ===
# Skew = some partitions contain far more data than others
# Symptom: a few tasks take much longer than the rest in the same stage

from pyspark.sql import Row
from pyspark.sql.functions import col, count

# Create a SKEWED dataset: 90% of records have category 'A'
skewed_data = (
    [Row(id=i, category='A')       for i in range(90_000)] +  # 90% skewed
    [Row(id=i, category=str(i%9))  for i in range(10_000)]    # 10% distributed
)
skewed_df = spark.createDataFrame(skewed_data)

print('Running aggregation on skewed data...')
print('Open Spark UI > Stages > Summary Metrics while this runs.')
print('Max task duration >> 75th percentile = skew detected\n')

result = skewed_df.groupBy('category').count().orderBy('count', ascending=False)
result.show()

print('\nRemedies for data skew:')
print('  1. Enable AQE (see next cell) -- handles skewed joins automatically')
print('  2. Salt skewed keys: add a random prefix, aggregate in two steps')
print('  3. Broadcast the smaller table in a join to avoid shuffle-based skew')

In [0]:
# === DEMO: Adaptive Query Execution (AQE) ===
# AQE dynamically optimises queries at runtime, including skewed joins

# Check AQE status
aqe_db = spark.conf.get('spark.databricks.optimizer.adaptive.enabled')
aqe_sp = spark.conf.get('spark.sql.adaptive.enabled')
print(f'spark.databricks.optimizer.adaptive.enabled: {aqe_db}')
print(f'spark.sql.adaptive.enabled:                  {aqe_sp}')

print("""
AQE handles three key optimisations automatically:
  1. Skewed join optimisation  -- detects and splits large partitions at runtime
  2. Dynamic partition coalescing -- merges small shuffle partitions after a stage
  3. Join strategy switch -- converts sort-merge joins to broadcast joins when
     a table turns out to be small enough during execution

In the Spark UI (SQL/DataFrame tab), AQE-optimised plans show:
  'AQEShuffleRead' and 'CustomShuffleReader' nodes
""")

In [0]:
# === DEMO: Memory spill -- investigation and prevention ===

# Check current shuffle partition setting
current = spark.conf.get('spark.sql.shuffle.partitions')
print(f'Current spark.sql.shuffle.partitions: {current}')

# Set to 'auto' for AQE-driven dynamic optimisation
spark.conf.set('spark.sql.shuffle.partitions', 'auto')
print(f'After set to auto:                    {spark.conf.get("spark.sql.shuffle.partitions")}')

print("""
To investigate spill in the Spark UI:
  1. Spark UI > Stages > click a stage involving aggregation or sort
  2. At the top of the stage page look for:
       Shuffle Spill (Memory) -- data spilled from RAM to off-heap
       Shuffle Spill (Disk)   -- data written to disk due to memory pressure
  3. Non-zero values indicate memory pressure

Remedies for spill:
  - Increase partition count (spark.sql.shuffle.partitions)
  - Use workers with higher memory-to-core ratios
  - Select only needed columns early (column pruning) to reduce data volume
  - Fix data skew first -- skew and spill often go together
""")

In [0]:
# === DEMO: Shuffle investigation and broadcast join optimisation ===
# Shuffle moves data between nodes during joins, aggregations, repartitioning

from pyspark.sql.functions import broadcast, col

# Large fact table and small dimension table
large_df = (spark.range(0, 500_000)
              .toDF('id')
              .withColumn('category', (col('id') % 50).cast('string')))

small_df = spark.createDataFrame(
    [(str(i), f'Category-{i}') for i in range(50)],
    ['category', 'category_name']
)

print('=== WITHOUT broadcast hint -- sort-merge join (causes shuffle) ===')
no_broadcast = large_df.join(small_df, on='category', how='left')
no_broadcast.explain(mode='simple')

print('\n=== WITH broadcast hint -- broadcast hash join (no shuffle) ===')
with_broadcast = large_df.join(broadcast(small_df), on='category', how='left')
with_broadcast.explain(mode='simple')

# Execute the broadcast version
with_broadcast.count()
print('\nCompare the query plans above.')
print('Look for SortMergeJoin (shuffle) vs. BroadcastHashJoin (no shuffle).')
print('Spark UI > SQL/DataFrame > DAG: Exchange nodes indicate shuffle.')

## Azure Log Analytics를 사용하여 로그 스트리밍 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/monitor-troubleshoot-optimize-workloads-azure-databricks.implement-log-streaming-azure-analytics.png)

### 로그 스트리밍 아키텍처 이해
> **트레이너 데모 가이드** -- 일회성 설정(플랫폼 관리자)
>> 1. **Azure Portal** 열기 > Azure Databricks 작업 영역 리소스로 이동
>> 2. **모니터링**에서 **진단 설정**을 클릭합니다.
>> 3. **+ 진단 설정 추가**를 클릭합니다.
>> 4. 로그 카테고리 선택: `clusters`, `jobs`, `notebook`, `SQL`, `workspace`, `unityCatalog`
>> 5. 대상을 **Log Analytics 작업 영역**으로 설정합니다.
>> 6. 저장 - 로그는 약 15분 내에 Log Analytics에 표시됩니다.
>> **데이터 흐름:**
>> 
>> Azure Databricks 이벤트
>> --> Azure 진단 설정
>> --> Log Analytics 작업 영역(서비스별 테이블)
>> --> KQL 쿼리/경고/대시보드
>> 
>> **전제조건:**
>> - Azure Databricks **프리미엄 플랜**(진단 로그에 필요)
>> - Azure의 Log Analytics 작업 영역
>> **캡처 대상:** 클러스터 수명 주기, 작업 실행/실패, 노트북 실행,
>> Unity 카탈로그 데이터 액세스, 비밀 액세스 이벤트

---

#### 결과 테이블 정렬 방법
- Log Analytics 쿼리 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 오름차순 또는 내림차순으로 정렬할 수 있습니다.
- KQL에서는 `order by` 절을 사용하여 원하는 기준으로 정렬할 수 있습니다.
- Databricks SQL에서는 `ORDER BY` 또는 PySpark에서는 `.orderBy()` 또는 `.sort()` 메서드를 사용하여 정렬할 수 있습니다.

### Azure Databricks 로그 테이블 살펴보기
Log Analytics는 Azure Databricks 로그를 목적별 테이블로 구성합니다.
 Table | Description |
-------|-------------|
 `DatabricksClusters` | Cluster creation, termination, resizing, configuration changes |
 `DatabricksJobs` | Job creation, runs, failures, schedule modifications |
 `DatabricksNotebook` | Notebook execution, creation, and modification events |
 `DatabricksSQL` | SQL warehouse operations and query events |
 `DatabricksUnityCatalog` | Catalog, schema, and table access events |
 `DatabricksWorkspace` | Workspace-level and administrative actions |
 `DatabricksSecrets` | Secret scope and secret access events |

**모든 테이블의 표준 열:**  
- `TimeGenerated` -- 이벤트가 발생한 시간  
- `OperationName` -- 수행된 작업  
- `Identity` -- 사용자 또는 서비스 주체  
- `SourceIPAddress` -- 요청이 시작된 곳  
- `RequestParams` -- 요청에 대한 세부정보  
- `Response` -- 상태 코드를 포함한 결과  

#### 결과 테이블 정렬 방법
- Log Analytics 쿼리 결과 테이블에서 열 이름 위에 마우스를 올리고 정렬 아이콘을 클릭하여 오름차순 또는 내림차순으로 정렬할 수 있습니다.
- KQL에서는 `order by` 절을 사용하여 원하는 기준으로 정렬할 수 있습니다.
- Databricks SQL에서는 `ORDER BY` 또는 PySpark에서는 `.orderBy()` 또는 `.sort()` 메서드를 사용하여 정렬할 수 있습니다.

> **팁:** 자주 사용하는 쿼리를 Log Analytics의 **쿼리 탐색기**에 저장하세요. 폴더로 정리하고 팀과 공유하세요.

### KQL(Kusto 쿼리 언어)을 사용한 쿼리 로그
**Azure Portal**에서 Log Analytics 작업 영역으로 이동하여 **로그**를 엽니다.
**최근 작업 이벤트(지난 24시간):**```kusto
DatabricksJobs
| where TimeGenerated > ago(24h)
| project TimeGenerated, OperationName, Identity, SourceIPAddress
| order by TimeGenerated desc
| take 100
```

**작업 실패 패턴 식별:**```kusto
DatabricksJobs
| where TimeGenerated > ago(7d)
| where Response contains "error" or Response contains "failed"
| summarize FailureCount = count() by OperationName, bin(TimeGenerated, 1h)
| order by FailureCount desc
```

**클러스터 사용 이벤트 추적:**```kusto
DatabricksClusters
| where TimeGenerated > ago(7d)
| summarize EventCount = count() by OperationName
| order by EventCount desc
```

**Unity Catalog 액세스 모니터링(보안 및 규정 준수):**```kusto
DatabricksUnityCatalog
| where TimeGenerated > ago(24h)
| where OperationName has "getTable" or OperationName has "selectFromTable"
| project TimeGenerated, Identity, OperationName, RequestParams
| order by TimeGenerated desc
```

---

### 사전 모니터링을 위한 알림 생성
> **트레이너 데모 가이드**
>> 1. Log Analytics에서 KQL 쿼리 실행
>> 2. 툴바에서 **새 경고 규칙**을 클릭합니다.
>> 3. **조건**을 구성합니다(예: FailureCount > 시간당 5회).
>> 4. **작업 그룹** 정의: 이메일, SMS, 웹훅(Teams/Slack)
>> 5. 경고 규칙 이름, 심각도 및 검토 빈도 설정
>> 6. **검토 + 만들기**를 클릭하세요.
>> **정렬 팁:** KQL 쿼리에서 `order by` 절을 사용하여 결과를 원하는 기준으로 정렬할 수 있습니다. 예시: `order by FailureCount desc`
>> **사용 사례 예:**
>> - 작업 실패율이 임계값을 초과하는 경우 경고
>> - Unity 카탈로그의 비정상적인 데이터 액세스 패턴에 대해 알림
>> - 중요한 클러스터가 예기치 않게 종료되면 웹후크를 트리거합니다.